# Credit Risk Modeling

This notebook develops and evaluates logistic regression models for the Give Me Some Credit dataset.

1. Load and preprocess the data
2. Establish a baseline logistic regression
3. Evaluate baseline AIC/BIC, VIF, and coefficient significance
4. Create candidate engineered features
5. Test candidate features individually against the baseline
6. Select promising features and evaluate combined models
7. Define and evaluate a final feature set
8. Proceed to model validation

## 1. Imports and Configuration

In [2]:
from pathlib import Path
import sys
import importlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

import src.features
import src.model
import src.preprocessing
from src.data_loader import load_data

importlib.reload(src.preprocessing)
importlib.reload(src.model)
importlib.reload(src.features)

<module 'src.features' from 'c:\\Users\\htuns\\Desktop\\credit-risk-model\\src\\features.py'>

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cs-training.csv"

TARGET = "SeriousDlqin2yrs"

BASELINE_FEATURES = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
]

## 2. Load and Preprocess Data

The preprocessing below follows the decisions established during EDA:

- train/test split
- remove invalid ages
- handle sentinel values
- handle extreme debt-ratio observations
- cap extreme revolving utilization
- median impute `MonthlyIncome` and `NumberOfDependents`

In [4]:
df = load_data(DATA_PATH)

train_df, test_df = src.preprocessing.split_data(df,test_size=0.2,random_state=67)

train_df = src.preprocessing.remove_invalid_ages(train_df)
test_df = src.preprocessing.remove_invalid_ages(test_df)

train_df = src.preprocessing.handle_sentinel_values(train_df)
test_df = src.preprocessing.handle_sentinel_values(test_df)

train_df, test_df = src.preprocessing.handle_debt_ratio_outlier(train_df, test_df)

train_df, test_df = src.preprocessing.handle_revolving_utilization_outlier(train_df, test_df)

train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="MonthlyIncome")
train_df, test_df = src.preprocessing.impute_missing(train_df=train_df, 
                                       test_df=test_df, 
                                       col="NumberOfDependents")



print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")

Train shape: (119792, 13)
Test shape:  (29938, 13)


## 3. Baseline Model

In [5]:
baseline_model = src.model.fit_logistic_model(train_df, TARGET, BASELINE_FEATURES)

baseline_train_prob = src.model.predict_probabilities(baseline_model, train_df, BASELINE_FEATURES)
baseline_test_prob = src.model.predict_probabilities(baseline_model, test_df, BASELINE_FEATURES)

baseline_train_metrics = src.model.evaluate_model(train_df[TARGET], baseline_train_prob)

baseline_test_metrics = src.model.evaluate_model(
    test_df[TARGET], baseline_test_prob)

print("Baseline predictive performance")
print(f"Train: {baseline_train_metrics}")
print(f"Test:  {baseline_test_metrics}")
print(
    f"ROC-AUC Train-Test Gap: "
    f"{baseline_train_metrics['ROC-AUC'] - baseline_test_metrics['ROC-AUC']:.4f}"
)
print(
    f"KS Train-Test Gap: "
    f"{baseline_train_metrics['KS'] - baseline_test_metrics['KS']:.4f}"
)

Baseline predictive performance
Train: {'ROC-AUC': 0.84932, 'KS': 0.53997}
Test:  {'ROC-AUC': 0.85196, 'KS': 0.54577}
ROC-AUC Train-Test Gap: -0.0026
KS Train-Test Gap: -0.0058


In [6]:
baseline_logit = src.model.fit_logit_inference(
    train_df, TARGET, BASELINE_FEATURES
)

baseline_ic = src.model.calc_information_criterion(baseline_logit)

print(f"AIC: {baseline_ic['AIC']:.2f}")
print(f"BIC: {baseline_ic['BIC']:.2f}")

Optimization terminated successfully.
         Current function value: 0.185706
         Iterations 8
AIC: 44514.29
BIC: 44620.92


In [7]:
baseline_inference = src.model.summarize_logit_inference(baseline_logit)
display(baseline_inference)

,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.320983,0.062321,0.000000e+00,-3.443131,-3.198835
RevolvingUtilizationOfUnsecuredLines,1.921069,0.036811,0.000000e+00,1.848919,1.993218
age,-0.018866,0.001055,1.824651e-71,-0.020934,-0.016797
NumberOfTime30-59DaysPastDueNotWorse,0.434841,0.012566,2.132923e-262,0.410213,0.459470
DebtRatio,0.086101,0.022085,9.671141e-05,0.042816,0.129386
MonthlyIncome,-0.000023,0.000004,1.310890e-10,-0.000030,-0.000016
NumberOfOpenCreditLinesAndLoans,0.028346,0.002949,7.032822e-22,0.022566,0.034125
NumberOfTimes90DaysLate,0.664725,0.018706,1.387226e-276,0.628063,0.701388
NumberRealEstateLoansOrLines,0.071273,0.012302,6.882716e-09,0.047162,0.095384
NumberOfTime60-89DaysPastDueNotWorse,0.607559,0.025970,4.812588e-121,0.556659,0.658458


In [8]:
baseline_vif = src.model.calculate_vif(train_df, BASELINE_FEATURES)
display(baseline_vif)

,feature,VIF
0,const,21.239728
1,RevolvingUtilizationOfUnsecuredLines,1.229571
2,age,1.146800
3,NumberOfTime30-59DaysPastDueNotWorse,1.181042
4,DebtRatio,1.103997
5,MonthlyIncome,1.028360
6,NumberOfOpenCreditLinesAndLoans,1.304639
7,NumberOfTimes90DaysLate,1.164158
8,NumberRealEstateLoansOrLines,1.318759
9,NumberOfTime60-89DaysPastDueNotWorse,1.187961


## 4. Feature Engineering

Candidate features are created in `src/features.py`.

The notebook does not automatically include every engineered feature in the model. It creates a feature set explicitly for each experiment:

```text
Baseline
Baseline + one candidate
Baseline + selected candidates
```

In [9]:
train_features, test_features = src.features.create_train_test_features(train_df, test_df)

engineered_features = [col for col in train_features.columns 
                       if col not in train_df.columns
]

print(f"Original columns:   {train_df.shape[1]}")
print(f"Engineered columns: {train_features.shape[1]}")
print("\nEngineered features:")
print(engineered_features)

Original columns:   13
Engineered columns: 44

Engineered features:
['TotalPastDue', 'AnyPastDue', 'Any30Plus', 'Any60Plus', 'Any90Plus', 'MultipleDelinquencies', 'PastDueSeverity', 'LogMonthlyIncome', 'RealEstateLoanShare', 'HighUtilization_50', 'HighUtilization_75', 'HighUtilization_90', 'MaxedOutUtilization', 'MultiplePastDue_2Plus', 'MultiplePastDue_3Plus', 'SevereDelinquency', 'RepeatedSevereDelinquency', 'YoungBorrower', 'OlderBorrower', 'ManyOpenCreditLines', 'HighRealEstateLoans', 'Utilization_x_TotalPastDue', 'DebtRatio_x_LogIncome', 'Age_x_Utilization', 'HighUtilization_x_AnyPastDue', 'SevereDelinquency_x_HighUtilization', 'AgeBin', 'UtilizationBin', 'DebtRatioBin', 'EstimatedDebt', 'IncomePerDependent']


### Information Value Screening

Rank baseline and engineered features by Information Value (IV) before selecting candidates for testing.

IV gives a model-agnostic metric of each feature's univariate separation power and is used here as supporting evidence for candidate selection in Section 7, alongside the p-value/VIF criteria applied later on.

In [10]:
import importlib
importlib.reload(src.model)

test_data = train_features[["NumberOfTimes90DaysLate", TARGET]].copy()
print(test_data["NumberOfTimes90DaysLate"].nunique())
print(test_data["NumberOfTimes90DaysLate"].value_counts())

iv, woe_table = src.model.calculate_woe_iv(
    train_features,
    feature="NumberOfTimes90DaysLate",
    target=TARGET,
)
print(iv)
display(woe_table)

17
NumberOfTimes90DaysLate
0     113321
1       4199
2       1263
3        536
4        231
5         95
6         70
7         27
9         17
8         16
10         7
12         2
11         2
13         2
14         2
17         1
15         1
Name: count, dtype: int64
0.8495467188773116


,bin,total,bad,good,dist_good,dist_bad,woe,iv
0,0,113321,5241,108080,0.965923,0.663502,0.375552,0.113575
1,1,4199,1405,2794,0.024970,0.177871,-1.963370,0.300200
2,10,7,5,2,0.000018,0.000633,-3.567098,0.002194
3,11,2,2,0,0.000001,0.000253,-5.534166,0.001396
4,12,2,1,1,0.000009,0.000127,-2.650807,0.000312
5,13,2,0,2,0.000018,0.000001,2.883359,0.000049
6,14,2,1,1,0.000009,0.000127,-2.650807,0.000312
7,15,1,0,1,0.000009,0.000001,2.190212,0.000017
8,17,1,1,0,0.000001,0.000127,-4.841019,0.000608
9,2,1263,630,633,0.005657,0.079757,-2.646056,0.196072


In [11]:
iv_screening_features = [
    col for col in train_features.columns
    if col != TARGET
]

iv_table = src.model.calculate_iv_all_features(
    train_features,
    target=TARGET,
    features=iv_screening_features,
)

display(iv_table)

,feature,IV
0,Utilization_x_TotalPastDue,3.407643
1,PastDueSeverity,1.512943
2,TotalPastDue,1.426285
3,Any30Plus,1.159350
4,AnyPastDue,1.159350
5,UtilizationBin,1.117221
6,RevolvingUtilizationOfUnsecuredLines,1.088131
7,Any60Plus,1.046124
8,MultipleDelinquencies,1.011952
9,MultiplePastDue_2Plus,1.011952


### Interpretation

Standard IV thresholds:

```text
< 0.02       Not useful for prediction
0.02 - 0.1   Weak predictive power
0.1  - 0.3   Medium predictive power
0.3  - 0.5   Strong predictive power
> 0.5        Suspicious (check for target leakage)
```

### Candidate feature groups

In [12]:
DELINQUENCY_FEATURES = [
    "AnyPastDue",
    "SevereDelinquency",
    "PastDueSeverity",
]

UTILIZATION_FEATURES = [
    "HighUtilization_75",
    "HighUtilization_90",
    "MaxedOutUtilization",
]

INTERACTION_FEATURES = [
    "HighUtilization_x_AnyPastDue",
    "SevereDelinquency_x_HighUtilization",
    "Utilization_x_TotalPastDue",
]

TRANSFORMATION_FEATURES = [
    "DebtRatio_x_LogIncome",
    "LogMonthlyIncome",
]

CANDIDATE_FEATURES = (
    DELINQUENCY_FEATURES
    + UTILIZATION_FEATURES
    + INTERACTION_FEATURES
    + TRANSFORMATION_FEATURES
    + ["EstimatedDebt"]
    + ["IncomePerDependent"]
)

missing_candidates = [
    feature for feature in CANDIDATE_FEATURES
    if feature not in train_features.columns
]

if missing_candidates:
    print("Candidate features not yet present in src/features.py:")
    for feature in missing_candidates:
        print(f"  - {feature}")
else:
    print("All candidate features are available.")

All candidate features are available.


## 5. Model Evaluation

`evaluate_feature_set()` is the main evaluation function for the notebook. For a specified feature set, it evaluates:

- train/test ROC-AUC and KS
- train/test train-test gaps
- AIC and BIC
- coefficient estimates, p-values, and confidence intervals
- VIF

`sklearn` is used for predictive performance and `statsmodels` is used for inference and information criteria.


In [13]:
def evaluate_feature_set(train_df, test_df, features, target_col=TARGET):
    """
    Fit and evaluate a logistic regression feature set.

    Returns predictive performance, train/test gaps, AIC/BIC,
    inference results, VIF, and the fitted models.
    """

    model = src.model.fit_logistic_model(train_df, target_col, features)

    train_prob = src.model.predict_probabilities(model, train_df, features)
    test_prob = src.model.predict_probabilities(model, test_df, features)

    train_metrics = src.model.evaluate_model(train_df[target_col], train_prob)
    test_metrics = src.model.evaluate_model(test_df[target_col], test_prob)

    result = src.model.fit_logit_inference(train_df, target_col, features)
    information_criteria = src.model.calc_information_criterion(result)
    inference = src.model.summarize_logit_inference(result)
    vif = src.model.calculate_vif(train_df, features)

    return {
        "features": features,
        "model": model,
        "statsmodels_result": result,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "train_test_gaps": {
            "ROC-AUC": round(train_metrics["ROC-AUC"] - test_metrics["ROC-AUC"], 5),
            "KS": round(train_metrics["KS"] - test_metrics["KS"], 5),
        },
        "information_criteria": information_criteria,
        "inference": inference,
        "vif": vif,
    }


def print_feature_set_results(name, result):
    """Print the main evaluation statistics for one feature set."""

    print(name)
    print("=" * len(name))
    print(f"Train: {result['train_metrics']}")
    print(f"Test:  {result['test_metrics']}")
    print(f"ROC-AUC Train-Test Gap: {result['train_test_gaps']['ROC-AUC']:.5f}")
    print(f"KS Train-Test Gap: {result['train_test_gaps']['KS']:.5f}")
    print(f"AIC: {result['information_criteria']['AIC']:.2f}")
    print(f"BIC: {result['information_criteria']['BIC']:.2f}")


## 6. `TotalPastDue`: Replacement Test

`TotalPastDue` is tested separately because it **replaces** the three original delinquency variables.

The purpose is to determine whether compressing the three variables into one aggregate loses useful information.

In [14]:
PAST_DUE_FEATURES = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

TOTAL_PAST_DUE_FEATURES = [
    feature for feature in BASELINE_FEATURES
    if feature not in PAST_DUE_FEATURES
] + ["TotalPastDue"]

total_past_due_result = evaluate_feature_set(
    train_features,
    test_features,
    TOTAL_PAST_DUE_FEATURES,
)

print_feature_set_results("TotalPastDue replacement", total_past_due_result)

print("\nInference:")
display(total_past_due_result["inference"])

print("VIF:")
display(total_past_due_result["vif"])


Optimization terminated successfully.
         Current function value: 0.186165
         Iterations 8
TotalPastDue replacement
Train: {'ROC-AUC': 0.84798, 'KS': 0.53907}
Test:  {'ROC-AUC': 0.85067, 'KS': 0.54306}
ROC-AUC Train-Test Gap: -0.00269
KS Train-Test Gap: -0.00399
AIC: 44620.09
BIC: 44707.33

Inference:


,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.278825,0.061966,0.000000e+00,-3.400276,-3.157374
RevolvingUtilizationOfUnsecuredLines,1.929470,0.036659,0.000000e+00,1.857621,2.001320
age,-0.018898,0.001052,3.514435e-72,-0.020959,-0.016836
DebtRatio,0.081751,0.022287,2.444172e-04,0.038068,0.125433
MonthlyIncome,-0.000023,0.000004,8.432919e-11,-0.000030,-0.000016
NumberOfOpenCreditLinesAndLoans,0.022700,0.002932,9.700370e-15,0.016954,0.028446
NumberRealEstateLoansOrLines,0.067434,0.012350,4.752573e-08,0.043229,0.091640
NumberOfDependents,0.035728,0.011154,1.358970e-03,0.013867,0.057589
TotalPastDue,0.530296,0.008067,0.000000e+00,0.514486,0.546107


VIF:


,feature,VIF
0,const,21.223460
1,RevolvingUtilizationOfUnsecuredLines,1.228397
2,age,1.146781
3,DebtRatio,1.103643
4,MonthlyIncome,1.028348
5,NumberOfOpenCreditLinesAndLoans,1.290095
6,NumberRealEstateLoansOrLines,1.317960
7,NumberOfDependents,1.080967
8,TotalPastDue,1.122885


### Interpretation

`TotalPastDue` is rejected as a replacement for the three individual delinquency variables because it produced slightly lower test ROC-AUC and KS.

The original three variables also had very low VIFs, so multicollinearity does not justify replacing them with the aggregate.

The next step is therefore to test alternative representations such as `AnyPastDue`, `SevereDelinquency`, and `PastDueSeverity` while retaining the original delinquency variables.

## 7. Candidate Feature Screening

Test each candidate **individually** against the same baseline.

```text
Baseline + AnyPastDue
Baseline + SevereDelinquency
Baseline + PastDueSeverity
```

This isolates the incremental contribution of each engineered feature.

In [15]:
def screen_candidate_features(
    train_df,
    test_df,
    candidate_features,
    baseline_features=BASELINE_FEATURES,
    target_col=TARGET
):
    """
    Evaluate each candidate feature individually against the same baseline.

    Each candidate is added separately; candidates are not tested together.
    """

    results = {}

    for feature in candidate_features:
        features = baseline_features + [feature]

        results[feature] = evaluate_feature_set(
            train_df,
            test_df,
            features,
            target_col,
        )

    return results


def summarize_feature_results(results, baseline_result=None):
    """
    Convert screening results into a comparison DataFrame.

    If baseline_result is supplied, changes relative to the baseline are
    included for test ROC-AUC, test KS, AIC, and BIC.
    """

    rows = []

    for feature, result in results.items():
        inference = result["inference"]
        vif = result["vif"]

        feature_row = inference.loc[feature]
        vif_rows = vif.loc[vif["feature"] == feature, "VIF"]

        row = {
            "Feature": feature,
            "Train ROC-AUC": result["train_metrics"]["ROC-AUC"],
            "Test ROC-AUC": result["test_metrics"]["ROC-AUC"],
            "Train KS": result["train_metrics"]["KS"],
            "Test KS": result["test_metrics"]["KS"],
            "AIC": result["information_criteria"]["AIC"],
            "BIC": result["information_criteria"]["BIC"],
            "Coefficient": feature_row["coefficient"],
            "p-value": feature_row["p_value"],
            "VIF": vif_rows.iloc[0] if not vif_rows.empty else np.nan,
        }

        if baseline_result is not None:
            row["Delta Test ROC-AUC"] = row["Test ROC-AUC"] - baseline_result["test_metrics"]["ROC-AUC"]
            row["Delta Test KS"] = row["Test KS"] - baseline_result["test_metrics"]["KS"]
            row["Delta AIC"] = row["AIC"] - baseline_result["information_criteria"]["AIC"]
            row["Delta BIC"] = row["BIC"] - baseline_result["information_criteria"]["BIC"]

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values("Test ROC-AUC", ascending=False)
        .reset_index(drop=True)
    )


In [16]:
def available_features(candidate_features, df):
    """Return only candidate features that currently exist in the DataFrame."""
    return [feature for feature in candidate_features if feature in df.columns]

available_delinquency_features = available_features(DELINQUENCY_FEATURES, train_features)
available_utilization_features = available_features(UTILIZATION_FEATURES, train_features)
available_interaction_features = available_features(INTERACTION_FEATURES, train_features)
available_transformation_features = available_features(TRANSFORMATION_FEATURES, train_features)

print("Available candidate features:")
print("Delinquency:", available_delinquency_features)
print("Utilization:", available_utilization_features)
print("Interactions:", available_interaction_features)
print("Transformations:", available_transformation_features)


Available candidate features:
Delinquency: ['AnyPastDue', 'SevereDelinquency', 'PastDueSeverity']
Utilization: ['HighUtilization_75', 'HighUtilization_90', 'MaxedOutUtilization']
Interactions: ['HighUtilization_x_AnyPastDue', 'SevereDelinquency_x_HighUtilization', 'Utilization_x_TotalPastDue']
Transformations: ['DebtRatio_x_LogIncome', 'LogMonthlyIncome']


In [17]:
# Use the same evaluator for the baseline so every experiment
# is compared against the same reference metrics.
baseline_result = evaluate_feature_set(train_features,test_features,BASELINE_FEATURES)

print_feature_set_results("Baseline model", baseline_result)

print("\nBaseline inference:")
display(baseline_result["inference"])

print("Baseline VIF:")
display(baseline_result["vif"])

Optimization terminated successfully.
         Current function value: 0.185706
         Iterations 8
Baseline model
Train: {'ROC-AUC': 0.84932, 'KS': 0.53997}
Test:  {'ROC-AUC': 0.85196, 'KS': 0.54577}
ROC-AUC Train-Test Gap: -0.00264
KS Train-Test Gap: -0.00580
AIC: 44514.29
BIC: 44620.92

Baseline inference:


,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.320983,0.062321,0.000000e+00,-3.443131,-3.198835
RevolvingUtilizationOfUnsecuredLines,1.921069,0.036811,0.000000e+00,1.848919,1.993218
age,-0.018866,0.001055,1.824651e-71,-0.020934,-0.016797
NumberOfTime30-59DaysPastDueNotWorse,0.434841,0.012566,2.132923e-262,0.410213,0.459470
DebtRatio,0.086101,0.022085,9.671141e-05,0.042816,0.129386
MonthlyIncome,-0.000023,0.000004,1.310890e-10,-0.000030,-0.000016
NumberOfOpenCreditLinesAndLoans,0.028346,0.002949,7.032822e-22,0.022566,0.034125
NumberOfTimes90DaysLate,0.664725,0.018706,1.387226e-276,0.628063,0.701388
NumberRealEstateLoansOrLines,0.071273,0.012302,6.882716e-09,0.047162,0.095384
NumberOfTime60-89DaysPastDueNotWorse,0.607559,0.025970,4.812588e-121,0.556659,0.658458


Baseline VIF:


,feature,VIF
0,const,21.239728
1,RevolvingUtilizationOfUnsecuredLines,1.229571
2,age,1.146800
3,NumberOfTime30-59DaysPastDueNotWorse,1.181042
4,DebtRatio,1.103997
5,MonthlyIncome,1.028360
6,NumberOfOpenCreditLinesAndLoans,1.304639
7,NumberOfTimes90DaysLate,1.164158
8,NumberRealEstateLoansOrLines,1.318759
9,NumberOfTime60-89DaysPastDueNotWorse,1.187961


In [18]:
delinquency_results = screen_candidate_features(
    train_features,
    test_features,
    available_delinquency_features
)

if delinquency_results:
    delinquency_results_df = summarize_feature_results(
        delinquency_results,
        baseline_result
    )
    display(delinquency_results_df)
else:
    print("No delinquency candidate features are currently available.")


Optimization terminated successfully.
         Current function value: 0.181543
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.183683
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185706
         Iterations 8


c:\Users\htuns\Desktop\credit-risk-model\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,AIC,BIC,Coefficient,p-value,VIF,Delta Test ROC-AUC,Delta Test KS,Delta AIC,Delta BIC
0,AnyPastDue,0.85543,0.85676,0.55189,0.56090,43518.74,43635.06,1.158804,1.463391e-231,2.335899,0.00480,0.01513,-995.55,-985.86
1,SevereDelinquency,0.85225,0.85492,0.54866,0.54964,44031.45,44147.77,1.237123,2.210204e-121,2.707149,0.00296,0.00387,-482.84,-473.15
2,PastDueSeverity,0.84935,0.85199,0.54004,0.54548,44516.29,44632.61,0.242942,9.999989e-01,inf,0.00003,-0.00029,2.00,11.69


## 7.1 Utilization Feature Screening

Each available utilization candidate is tested individually against the baseline.


In [19]:
utilization_results = screen_candidate_features(
    train_features,
    test_features,
    available_utilization_features,
)

if utilization_results:
    utilization_results_df = summarize_feature_results(utilization_results, baseline_result)
    display(utilization_results_df)
else:
    print("No utilization candidate features are currently available.")


Optimization terminated successfully.
         Current function value: 0.185706
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185705
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185698
         Iterations 8


,Feature,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,AIC,BIC,Coefficient,p-value,VIF,Delta Test ROC-AUC,Delta Test KS,Delta AIC,Delta BIC
0,MaxedOutUtilization,0.84948,0.85212,0.54112,0.54653,44514.29,44630.61,0.076678,0.156102,1.169947,0.00016,0.00076,0.00,9.69
1,HighUtilization_90,0.84936,0.85200,0.54000,0.54514,44516.01,44632.33,-0.023399,0.598175,2.475086,0.00004,-0.00063,1.72,11.41
2,HighUtilization_75,0.84935,0.85195,0.54020,0.54549,44516.28,44632.60,0.005213,0.920420,3.616664,-0.00001,-0.00028,1.99,11.68


## 7.2 Interaction Feature Screening

Each available interaction candidate is tested individually against the baseline.


In [20]:
interaction_results = screen_candidate_features(
    train_features,
    test_features,
    available_interaction_features
)

if interaction_results:
    interaction_results_df = summarize_feature_results(interaction_results, baseline_result)
    display(interaction_results_df)
else:
    print("No interaction candidate features are currently available.")


Optimization terminated successfully.
         Current function value: 0.185304
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185556
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.184338
         Iterations 8


,Feature,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,AIC,BIC,Coefficient,p-value,VIF,Delta Test ROC-AUC,Delta Test KS,Delta AIC,Delta BIC
0,Utilization_x_TotalPastDue,0.85183,0.85415,0.54364,0.55165,44188.36,44304.68,-0.391325,5.584574e-75,5.578134,0.00219,0.00588,-325.93,-316.24
1,HighUtilization_x_AnyPastDue,0.84959,0.85221,0.54129,0.54574,44419.79,44536.11,0.404404,7.333057e-23,1.858463,0.00025,-0.00003,-94.50,-84.81
2,SevereDelinquency_x_HighUtilization,0.84909,0.85177,0.54060,0.54554,44480.26,44596.59,0.346301,1.496816e-09,1.977305,-0.00019,-0.00023,-34.03,-24.33


## 7.3 Transformation Feature Screening

Each available transformation candidate is tested individually against the baseline.


In [21]:
transformation_results = screen_candidate_features(
    train_features,
    test_features,
    available_transformation_features,
)

if transformation_results:
    transformation_results_df = summarize_feature_results(transformation_results, baseline_result)
    display(transformation_results_df)
else:
    print("No transformation candidate features are currently available.")


Optimization terminated successfully.
         Current function value: 0.185558
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185674
         Iterations 8


,Feature,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,AIC,BIC,Coefficient,p-value,VIF,Delta Test ROC-AUC,Delta Test KS,Delta AIC,Delta BIC
0,DebtRatio_x_LogIncome,0.84972,0.85243,0.54113,0.54551,44480.82,44597.15,0.087137,1.024430e-07,14.466969,0.00047,-0.00026,-33.47,-23.77
1,LogMonthlyIncome,0.84932,0.85214,0.54074,0.54641,44508.51,44624.83,0.040441,7.023481e-03,1.155036,0.00018,0.00064,-5.78,3.91


## 7.4 Additional Candidate Feature Screening

`EstimatedDebt` (DebtRatio × MonthlyIncome) and `IncomePerDependent`
(MonthlyIncome / (NumberOfDependents + 1)). Screened individually against the baseline as candidates.

In [22]:
ADDITIONAL_FEATURES = [
    "EstimatedDebt",
    "IncomePerDependent",
]

available_additional_features = available_features(ADDITIONAL_FEATURES, train_features)

print("Available candidate features:")
print("Additional:", available_additional_features)

additional_results = screen_candidate_features(
    train_features,
    test_features,
    available_additional_features,
)

if additional_results:
    additional_results_df = summarize_feature_results(additional_results, baseline_result)
    display(additional_results_df)
else:
    print("No additional candidate features are currently available.")

Available candidate features:
Additional: ['EstimatedDebt', 'IncomePerDependent']
Optimization terminated successfully.
         Current function value: 0.185680
         Iterations 8
Optimization terminated successfully.
         Current function value: 0.185689
         Iterations 8


,Feature,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,AIC,BIC,Coefficient,p-value,VIF,Delta Test ROC-AUC,Delta Test KS,Delta AIC,Delta BIC
0,IncomePerDependent,0.84926,0.85208,0.54063,0.54755,44512.02,44628.34,0.000018,0.037629,3.608129,0.00012,0.00178,-2.27,7.42
1,EstimatedDebt,0.84948,0.85197,0.54011,0.54515,44509.88,44626.20,0.000012,0.005913,1.646930,0.00001,-0.00062,-4.41,5.28


## 7.5 Inspect Individual Candidate Models

The screening tables are the main comparison. Full inference and VIF results remain stored in each results dictionary for closer inspection.


In [23]:
# Example after screening:
feature = "SevereDelinquency"
result = delinquency_results[feature]
display(result["inference"])
display(result["vif"])


,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.385940,0.062894,0.000000e+00,-3.509210,-3.262671
RevolvingUtilizationOfUnsecuredLines,1.855961,0.037155,0.000000e+00,1.783139,1.928784
age,-0.019049,0.001064,1.032254e-71,-0.021134,-0.016964
NumberOfTime30-59DaysPastDueNotWorse,0.406499,0.012701,9.435150e-225,0.381605,0.431393
DebtRatio,0.093931,0.022003,1.962650e-05,0.050807,0.137055
MonthlyIncome,-0.000021,0.000004,9.370254e-10,-0.000028,-0.000015
NumberOfOpenCreditLinesAndLoans,0.032747,0.002941,8.644366e-29,0.026982,0.038512
NumberOfTimes90DaysLate,0.203932,0.023913,1.487331e-17,0.157064,0.250801
NumberRealEstateLoansOrLines,0.076211,0.012326,6.284772e-10,0.052053,0.100369
NumberOfTime60-89DaysPastDueNotWorse,0.562840,0.025998,6.195551e-104,0.511885,0.613796


,feature,VIF
0,const,21.261237
1,RevolvingUtilizationOfUnsecuredLines,1.241833
2,age,1.146868
3,NumberOfTime30-59DaysPastDueNotWorse,1.194711
4,DebtRatio,1.104095
5,MonthlyIncome,1.028372
6,NumberOfOpenCreditLinesAndLoans,1.306781
7,NumberOfTimes90DaysLate,2.626464
8,NumberRealEstateLoansOrLines,1.319354
9,NumberOfTime60-89DaysPastDueNotWorse,1.192721


## 8. Feature Selection

Use the screening results to identify promising features.

Consider:

- higher test ROC-AUC
- higher test KS
- small train/test gap
- lower AIC
- lower BIC
- statistically meaningful coefficient
- acceptable VIF
- credit-risk interpretability

A feature that improves AUC by a tiny amount but adds complexity and has weak statistical evidence does not necessarily belong in the final model.

### Combined candidate model

In [24]:
SELECTED_ENGINEERED_FEATURES = [
    "AnyPastDue",
    # Rejected: HighUtilization_* (no lift, HighUtilization_90 p=0.598)
    # Rejected: DebtRatio_x_LogIncome (VIF=14.47, unacceptable multicollinearity)
    # "Utilization_x_TotalPastDue"
    # Rejected: Utilization_x_TotalPastDue (VIF=5.58, marginal lift)
    # Rejected: LogMonthlyIncome (VIF fine, but negligible lift)
    # Rejected: SevereDelinquency (weaker than AnyPastDue, inflates NumberOfTimes90DaysLate VIF to 2.63)
    # Rejected: EstimatedDebt 
    # Rejected: IncomePerDependent
]

CANDIDATE_MODEL_FEATURES = (
    BASELINE_FEATURES + SELECTED_ENGINEERED_FEATURES
)

print("Candidate model features:")
for feature in CANDIDATE_MODEL_FEATURES:
    print(f"  - {feature}")

Candidate model features:
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - AnyPastDue


In [25]:
if SELECTED_ENGINEERED_FEATURES:
    candidate_model_result = evaluate_feature_set(
        train_features,
        test_features,
        CANDIDATE_MODEL_FEATURES,
    )

    print("Candidate model")
    print(f"Train: {candidate_model_result['train_metrics']}")
    print(f"Test:  {candidate_model_result['test_metrics']}")
    print(candidate_model_result["information_criteria"])

    display(candidate_model_result["inference"])
    display(candidate_model_result["vif"])
else:
    print("No engineered features selected yet.")

Optimization terminated successfully.
         Current function value: 0.181543
         Iterations 8
Candidate model
Train: {'ROC-AUC': 0.85543, 'KS': 0.55189}
Test:  {'ROC-AUC': 0.85676, 'KS': 0.5609}
{'AIC': 43518.74, 'BIC': 43635.06}


,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.603863,0.063921,0.000000e+00,-3.729147,-3.478580
RevolvingUtilizationOfUnsecuredLines,1.760338,0.037456,0.000000e+00,1.686925,1.833751
age,-0.017588,0.001066,3.677224e-61,-0.019677,-0.015499
NumberOfTime30-59DaysPastDueNotWorse,0.173859,0.014282,4.317931e-34,0.145866,0.201851
DebtRatio,0.098747,0.021891,6.454702e-06,0.055842,0.141652
MonthlyIncome,-0.000021,0.000003,1.248019e-09,-0.000028,-0.000014
NumberOfOpenCreditLinesAndLoans,0.027612,0.002959,1.057396e-20,0.021811,0.033412
NumberOfTimes90DaysLate,0.468562,0.017932,1.683501e-150,0.433416,0.503709
NumberRealEstateLoansOrLines,0.077684,0.012178,1.779211e-10,0.053816,0.101551
NumberOfTime60-89DaysPastDueNotWorse,0.423653,0.024182,1.013960e-68,0.376258,0.471048


,feature,VIF
0,const,21.393274
1,RevolvingUtilizationOfUnsecuredLines,1.247322
2,age,1.149111
3,NumberOfTime30-59DaysPastDueNotWorse,2.021205
4,DebtRatio,1.104112
5,MonthlyIncome,1.028365
6,NumberOfOpenCreditLinesAndLoans,1.304981
7,NumberOfTimes90DaysLate,1.239274
8,NumberRealEstateLoansOrLines,1.319202
9,NumberOfTime60-89DaysPastDueNotWorse,1.229069


## 9. Final Feature Set

Once feature selection is complete, lock the final feature set before the final evaluation.

The test set should be treated as the final out-of-sample evaluation set rather than repeatedly optimized against.

In [26]:
FINAL_FEATURES = [
    *BASELINE_FEATURES,
    "AnyPastDue",
]

print(f"Number of final features: {len(FINAL_FEATURES)}")
for feature in FINAL_FEATURES:
    print(f"  - {feature}")

Number of final features: 11
  - RevolvingUtilizationOfUnsecuredLines
  - age
  - NumberOfTime30-59DaysPastDueNotWorse
  - DebtRatio
  - MonthlyIncome
  - NumberOfOpenCreditLinesAndLoans
  - NumberOfTimes90DaysLate
  - NumberRealEstateLoansOrLines
  - NumberOfTime60-89DaysPastDueNotWorse
  - NumberOfDependents
  - AnyPastDue


## 10. Final Model vs. Baseline

Run this section only after the final feature set has been selected.

Compare:

- ROC-AUC
- KS
- AIC
- BIC
- train/test gaps
- coefficient significance
- VIF

In [27]:
final_result = evaluate_feature_set(
    train_features,
    test_features,
    FINAL_FEATURES
)

comparison = pd.DataFrame({
    "Baseline": {
        "Train ROC-AUC": baseline_result["train_metrics"]["ROC-AUC"],
        "Test ROC-AUC": baseline_result["test_metrics"]["ROC-AUC"],
        "Train KS": baseline_result["train_metrics"]["KS"],
        "Test KS": baseline_result["test_metrics"]["KS"],
        "ROC-AUC Gap": baseline_result["train_test_gaps"]["ROC-AUC"],
        "KS Gap": baseline_result["train_test_gaps"]["KS"],
        "AIC": baseline_result["information_criteria"]["AIC"],
        "BIC": baseline_result["information_criteria"]["BIC"],
    },
    "Final": {
        "Train ROC-AUC": final_result["train_metrics"]["ROC-AUC"],
        "Test ROC-AUC": final_result["test_metrics"]["ROC-AUC"],
        "Train KS": final_result["train_metrics"]["KS"],
        "Test KS": final_result["test_metrics"]["KS"],
        "ROC-AUC Gap": final_result["train_test_gaps"]["ROC-AUC"],
        "KS Gap": final_result["train_test_gaps"]["KS"],
        "AIC": final_result["information_criteria"]["AIC"],
        "BIC": final_result["information_criteria"]["BIC"],
    },
}).T

comparison["Delta Test ROC-AUC"] = comparison["Test ROC-AUC"] - comparison.loc["Baseline", "Test ROC-AUC"]
comparison["Delta Test KS"] = comparison["Test KS"] - comparison.loc["Baseline", "Test KS"]
comparison["Delta AIC"] = comparison["AIC"] - comparison.loc["Baseline", "AIC"]

print("Baseline vs. Final Model")
display(comparison)

print("\nFinal model inference:")
display(final_result["inference"])

print("\nFinal model VIF:")
display(final_result["vif"])

Optimization terminated successfully.
         Current function value: 0.181543
         Iterations 8
Baseline vs. Final Model


,Train ROC-AUC,Test ROC-AUC,Train KS,Test KS,ROC-AUC Gap,KS Gap,AIC,BIC,Delta Test ROC-AUC,Delta Test KS,Delta AIC
Baseline,0.84932,0.85196,0.53997,0.54577,-0.00264,-0.00580,44514.29,44620.92,0.0000,0.00000,0.00
Final,0.85543,0.85676,0.55189,0.56090,-0.00133,-0.00901,43518.74,43635.06,0.0048,0.01513,-995.55



Final model inference:


,coefficient,std_error,p_value,ci_lower,ci_upper
const,-3.603863,0.063921,0.000000e+00,-3.729147,-3.478580
RevolvingUtilizationOfUnsecuredLines,1.760338,0.037456,0.000000e+00,1.686925,1.833751
age,-0.017588,0.001066,3.677224e-61,-0.019677,-0.015499
NumberOfTime30-59DaysPastDueNotWorse,0.173859,0.014282,4.317931e-34,0.145866,0.201851
DebtRatio,0.098747,0.021891,6.454702e-06,0.055842,0.141652
MonthlyIncome,-0.000021,0.000003,1.248019e-09,-0.000028,-0.000014
NumberOfOpenCreditLinesAndLoans,0.027612,0.002959,1.057396e-20,0.021811,0.033412
NumberOfTimes90DaysLate,0.468562,0.017932,1.683501e-150,0.433416,0.503709
NumberRealEstateLoansOrLines,0.077684,0.012178,1.779211e-10,0.053816,0.101551
NumberOfTime60-89DaysPastDueNotWorse,0.423653,0.024182,1.013960e-68,0.376258,0.471048



Final model VIF:


,feature,VIF
0,const,21.393274
1,RevolvingUtilizationOfUnsecuredLines,1.247322
2,age,1.149111
3,NumberOfTime30-59DaysPastDueNotWorse,2.021205
4,DebtRatio,1.104112
5,MonthlyIncome,1.028365
6,NumberOfOpenCreditLinesAndLoans,1.304981
7,NumberOfTimes90DaysLate,1.239274
8,NumberRealEstateLoansOrLines,1.319202
9,NumberOfTime60-89DaysPastDueNotWorse,1.229069


## 11. Conclusions

Only AnyPastDue produced a meaningful and defensible improvement in out of sample performance. Adding AnyPastDue dropped AIC by approximately 995 points relative to the baseline model, indicating a substantial improvement in fit that justified the added complexity of a single additional term. Multicollinearity remained well within acceptable bounds, with all VIFs under 2.4(no redundancy).

On the test set, the final model achieved ROC-AUC of ~0.857 and KS of ~0.561, both representing a meaningful improvement in discriminatory power over the baseline. The train/test performance gap remained small, suggesting the model generalizes well and is not overfit to the training data.

The final feature set baseline variables + AnyPastDue was selected because it was the only configuration that cleared both bars required for inclusion: statistical significance and domain defensibility.